In [16]:
%%writefile tictactoe.py
import argparse
import random
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

SEED_VALUE = 24
EMPTY_BOARD = (0,) * 9
WIN_TRIPLETS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6)
]


def find_valid_moves(state_tuple):
    return [idx for idx, cell in enumerate(state_tuple) if cell == 0]


def apply_move(state_tuple, slot_idx, player_id):
    board_list = list(state_tuple)
    board_list[slot_idx] = player_id
    return tuple(board_list)


def determine_winner(state_tuple):
    for pos1, pos2, pos3 in WIN_TRIPLETS:
        if state_tuple[pos1] != 0 and state_tuple[pos1] == state_tuple[pos2] == state_tuple[pos3]:
            return state_tuple[pos1]
    if all(cell != 0 for cell in state_tuple):
        return 0
    return None


def swap_perspective(state_tuple):
    return tuple(0 if cell == 0 else 3 - cell for cell in state_tuple)


def render_board_str(state_tuple):
    char_map = ("·", "X", "O")
    grid_lines = [" | ".join(char_map[state_tuple[3 * r + c]] for c in range(3)) for r in range(3)]
    return "\n---------\n".join(grid_lines)


def get_state_q_values(q_matrix, state_tuple):
    return q_matrix.get(state_tuple, np.zeros(9))


def choose_eps_greedy_move(q_matrix, state_tuple, epsilon, random_gen):
    available = find_valid_moves(state_tuple)
    if random_gen.random() < epsilon:
        return random_gen.choice(available)
    q_vals = get_state_q_values(q_matrix, state_tuple)
    highest_q = max(q_vals[move] for move in available)
    best_candidate_moves = [move for move in available if q_vals[move] == highest_q]
    return random_gen.choice(best_candidate_moves)


def choose_best_move(q_matrix, state_tuple, random_gen):
    return choose_eps_greedy_move(q_matrix, state_tuple, 0.0, random_gen)


def run_self_play_epoch(q_matrix, epsilon, lr, discount, random_gen):
    current_board = EMPTY_BOARD
    current_player = 1
    recent_history = {1: None, 2: None}

    while True:
        view = current_board if current_player == 1 else swap_perspective(current_board)

        if recent_history[current_player] is not None:
            prev_s, prev_a = recent_history[current_player]
            valid_next = find_valid_moves(view)
            next_q = get_state_q_values(q_matrix, view)
            max_future = discount * max(next_q[a] for a in valid_next)
            q_matrix[prev_s][prev_a] += lr * (max_future - q_matrix[prev_s][prev_a])

        chosen_slot = choose_eps_greedy_move(q_matrix, view, epsilon, random_gen)
        recent_history[current_player] = (view, chosen_slot)

        current_board = apply_move(current_board, chosen_slot, current_player)
        result = determine_winner(current_board)

        if result is not None:
            if result == current_player:
                r_active, r_opponent = +1.0, -1.0
            else:
                r_active, r_opponent = +0.5, +0.5

            s_act, a_act = recent_history[current_player]
            q_matrix[s_act][a_act] += lr * (r_active - q_matrix[s_act][a_act])

            opp_player = 3 - current_player
            if recent_history[opp_player] is not None:
                s_opp, a_opp = recent_history[opp_player]
                q_matrix[s_opp][a_opp] += lr * (r_opponent - q_matrix[s_opp][a_opp])
            return result

        current_player = 3 - current_player


def simulate_random_match(q_matrix, agent_turn_id, random_gen):
    current_board = EMPTY_BOARD
    active = 1
    while True:
        if active == agent_turn_id:
            view = current_board if active == 1 else swap_perspective(current_board)
            slot = choose_best_move(q_matrix, view, random_gen)
        else:
            slot = random_gen.choice(find_valid_moves(current_board))

        current_board = apply_move(current_board, slot, active)
        result = determine_winner(current_board)
        if result is not None:
            return result
        active = 3 - active


def eval_against_baseline(q_matrix, total_games, random_gen):
    total_score = 0.0
    for match in range(total_games):
        role = 1 if match < total_games // 2 else 2
        outcome = simulate_random_match(q_matrix, role, random_gen)
        if outcome == role:
            total_score += 1.0
        elif outcome == 0:
            total_score += 0.5
    return total_score / total_games


def train_agent(num_epochs=35000, lr=0.3, discount=0.9,
                eps_init=1.0, eps_delta=0.2, eps_step=1000, eps_floor=0.0,
                eval_freq=1000, test_count=10, seed=SEED_VALUE):
    rng = random.Random(seed)
    q_table = defaultdict(lambda: np.zeros(9, dtype=np.float64))
    curr_epsilon = eps_init
    progress_log = []

    print(f"Training agent for {num_epochs} epochs...")
    for ep in range(1, num_epochs + 1):
        run_self_play_epoch(q_table, curr_epsilon, lr, discount, rng)

        if ep % eps_step == 0:
            curr_epsilon = max(eps_floor, curr_epsilon - eps_delta)

        if ep % eval_freq == 0:
            score = eval_against_baseline(q_table, test_count, rng)
            progress_log.append((ep, score))
            print(f"Epochs {ep:4d} | Evaluated Score: {score:.2f} | States: {len(q_table)}")

    return q_table, progress_log


def save_performance_plot(history_data, file_path="training_curve.png"):
    epochs, scores = zip(*history_data)
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, scores, marker='o', alpha=0.6)
    plt.xlabel("Epochs")
    plt.ylabel("Score vs Random")
    plt.title("Training Curve")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(file_path, dpi=120)
    plt.close()
    print(f"Plot saved to '{file_path}'.")


def play_human_vs_agent(q_table, num_matches=10):
    print("\n--- Human vs Agent Session ---")
    rng = random.Random()
    h_wins = a_wins = ties = 0

    for match_num in range(num_matches):
        human_id = 2 if match_num % 2 == 0 else 1
        agent_id = 3 - human_id
        print(f"\nGame {match_num + 1} (You are {'X' if human_id == 1 else 'O'})")

        board = EMPTY_BOARD
        turn = 1
        while True:
            if turn == agent_id:
                view = board if turn == 1 else swap_perspective(board)
                move = choose_best_move(q_table, view, rng)
                print(f"Agent played position {move}")
            else:
                print(render_board_str(board))
                valid = find_valid_moves(board)
                move = None
                while move is None:
                    try:
                        val = int(input(f"Your move {valid}: "))
                        if val in valid:
                            move = val
                    except ValueError:
                        pass

            board = apply_move(board, move, turn)
            status = determine_winner(board)

            if status is not None:
                print(render_board_str(board))
                if status == human_id:
                    print("Result: You won!")
                    h_wins += 1
                elif status == agent_id:
                    print("Result: Agent won.")
                    a_wins += 1
                else:
                    print("Result: Draw.")
                    ties += 1
                break
            turn = 3 - turn

    print(f"\nFinal Record -> Wins: {h_wins}, Losses: {a_wins}, Draws: {ties}")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=35000)
    parser.add_argument("--output_graph", type=str, default="training_curve.png")
    parser.add_argument("--skip-human", action="store_true")
    args = parser.parse_args()

    np.random.seed(SEED_VALUE)
    random.seed(SEED_VALUE)

    q_matrix, history = train_agent(num_epochs=args.epochs)
    save_performance_plot(history, args.output_graph)

    if not args.skip_human:
        play_human_vs_agent(q_matrix)

if __name__ == "__main__":
    main()

Overwriting tictactoe.py


In [17]:
!python tictactoe.py

Training agent for 35000 epochs...
Epochs 1000 | Evaluated Score: 0.70 | States: 2680
Epochs 2000 | Evaluated Score: 0.70 | States: 3579
Epochs 3000 | Evaluated Score: 0.80 | States: 3918
Epochs 4000 | Evaluated Score: 0.80 | States: 4032
Epochs 5000 | Evaluated Score: 0.95 | States: 4051
Epochs 6000 | Evaluated Score: 0.85 | States: 4051
Epochs 7000 | Evaluated Score: 0.90 | States: 4051
Epochs 8000 | Evaluated Score: 0.90 | States: 4051
Epochs 9000 | Evaluated Score: 0.90 | States: 4051
Epochs 10000 | Evaluated Score: 0.90 | States: 4051
Epochs 11000 | Evaluated Score: 1.00 | States: 4051
Epochs 12000 | Evaluated Score: 1.00 | States: 4051
Epochs 13000 | Evaluated Score: 1.00 | States: 4051
Epochs 14000 | Evaluated Score: 1.00 | States: 4051
Epochs 15000 | Evaluated Score: 0.90 | States: 4051
Epochs 16000 | Evaluated Score: 0.90 | States: 4051
Epochs 17000 | Evaluated Score: 1.00 | States: 4051
Epochs 18000 | Evaluated Score: 0.90 | States: 4051
Epochs 19000 | Evaluated Score: 1.00 |